## Part 1: Data Loading and Profiling

In [1]:
from PyDI.io import load_parquet

# Define dataset paths
DATA_DIR = "../datasets/"

# Load Kaggle 380k dataset
kaggle380k = load_parquet(
    DATA_DIR + "kaggle380k.parquet",
    name="kaggle380k",
)

# Load Uber Eats dataset  
uber_eats = load_parquet(
    DATA_DIR + "uber_eats.parquet",
    name="uber_eats",
)

# Load Yelp dataset
yelp = load_parquet(
    DATA_DIR + "yelp.parquet",
    name="yelp",
)

### Basic Dataset Summary

In [2]:
# Display basic information
datasets = [kaggle380k, uber_eats, yelp]
names = ["Kaggle 380k", "Uber Eats", "Yelp"]

for df, name in zip(datasets, names):
    print(f"{name}:")
    print(f"  Records: {len(df):,}")
    print(f"  Attributes: {len(df.columns)}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Dataset name: {df.attrs.get('dataset_name', 'unknown')}")
    print()

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Kaggle 380k:
  Records: 380,358
  Attributes: 20
  Columns: ['kaggle380k_id', 'source', 'name', 'name_norm', 'website', 'map_url', 'phone_raw', 'phone_e164', 'address_line1', 'address_line2', 'street', 'house_number', 'city', 'state', 'postal_code', 'country', 'latitude', 'longitude', 'categories', 'rating']
  Dataset name: kaggle380k

Uber Eats:
  Records: 63,469
  Attributes: 17
  Columns: ['uber_eats_id', 'source', 'name', 'name_norm', 'address_line1', 'address_line2', 'street', 'house_number', 'city', 'state', 'postal_code', 'country', 'latitude', 'longitude', 'categories', 'rating', 'rating_count']
  Dataset name: uber_eats

Yelp:
  Records: 99,500
  Attributes: 20
  Columns: ['yelp_id', 'source', 'name', 'name_norm', 'map_url', 'phone_raw', 'phone_e164', 'address_line1', 'address_line2', 'street', 'house_number', 'city', 'state', 'postal_code', 'country', 'latitude', 'longitude', 'categories', 'rating', 'rating_count']
  Dataset name: yelp

Total records across all datasets: 543,

In [3]:
from PyDI.profiling import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

kaggle380k:
  Rows: 380,358
  Columns: 20
  Total nulls: 853,673
  Null percentage: 11.2%
  Null counts per column:
    website: 94,923 (25.0%)
    map_url: 35,149 (9.2%)
    phone_raw: 48,860 (12.8%)
    phone_e164: 52,341 (13.8%)
    address_line1: 17,684 (4.6%)
    address_line2: 331,013 (87.0%)
    street: 34,160 (9.0%)
    house_number: 36,100 (9.5%)
    city: 17,712 (4.7%)
    state: 69,929 (18.4%)
    postal_code: 72,884 (19.2%)
    latitude: 218 (0.1%)
    longitude: 218 (0.1%)
    rating: 42,482 (11.2%)

uber_eats:
  Rows: 63,469
  Columns: 17
  Total nulls: 119,668
  Null percentage: 11.1%
  Null counts per column:
    address_line1: 715 (1.1%)
    address_line2: 57,504 (90.6%)
    street: 2,237 (3.5%)
    house_number: 897 (1.4%)
    city: 583 (0.9%)
    state: 881 (1.4%)
    postal_code: 517 (0.8%)
    rating: 28,167 (44.4%)
    rating_count: 28,167 (44.4%)

yelp:
  Rows: 99,500
  Columns: 20
  Total nulls: 116,812
  Null percentage: 5.9%
  Null counts per column:
    phone

{'rows': 99500,
 'columns': 20,
 'nulls_total': 116812,
 'nulls_per_column': {'yelp_id': 0,
  'source': 0,
  'name': 0,
  'name_norm': 0,
  'map_url': 0,
  'phone_raw': 5543,
  'phone_e164': 5648,
  'address_line1': 2137,
  'address_line2': 98916,
  'street': 2162,
  'house_number': 2349,
  'city': 0,
  'state': 2,
  'postal_code': 51,
  'country': 0,
  'latitude': 2,
  'longitude': 2,
  'categories': 0,
  'rating': 0,
  'rating_count': 0},
 'dtypes': {'yelp_id': 'object',
  'source': 'object',
  'name': 'object',
  'name_norm': 'object',
  'map_url': 'object',
  'phone_raw': 'object',
  'phone_e164': 'object',
  'address_line1': 'object',
  'address_line2': 'object',
  'street': 'object',
  'house_number': 'object',
  'city': 'object',
  'state': 'object',
  'postal_code': 'object',
  'country': 'object',
  'latitude': 'float64',
  'longitude': 'float64',
  'categories': 'object',
  'rating': 'float64',
  'rating_count': 'int64'}}

### Attribute Coverage Analysis

In [4]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 3]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

📊 Attribute coverage across datasets:


,attribute,kaggle380k_count,kaggle380k_pct,kaggle380k_coverage,kaggle380k_samples,uber_eats_count,uber_eats_pct,uber_eats_coverage,uber_eats_samples,yelp_count,yelp_pct,yelp_coverage,yelp_samples,avg_coverage,max_coverage,datasets_with_attribute
0,address_line1,362674/380358,95.4%,0.953507,"['3143 US-280', '16 Broad St', '68 Broad St']",62754/63469,98.9%,0.988735,"['224 Daniel Payne Drive', '1521 Pinson Valley...",97363/99500,97.9%,0.978523,"['135-25 142nd St', '135-58 Lefferts Blvd', '1...",0.973588,0.988735,3
1,address_line2,49345/380358,13.0%,0.129733,"['Suite 5', '# 15', '# 5416']",5965/63469,9.4%,0.093983,"['5', 'Unit 101', 'Suite 123']",584/99500,0.6%,0.005869,"['Ter 8', '5', '1']",0.076528,0.129733,3
2,categories,380358/380358,100.0%,1.000000,"[['Fast food restaurant', 'Ice cream shop'], [...",63469/63469,100.0%,1.000000,"[['Burgers', 'American', 'Sandwiches'], ['Coff...",99500/99500,100.0%,1.000000,"[['breakfast'], ['italian', 'seafood'], ['ital...",1.000000,1.000000,3
3,city,362646/380358,95.3%,0.953433,"['Alexander City', 'Alexander City', 'Alexande...",62886/63469,99.1%,0.990814,"['Birmingham', 'Birmingham', 'Birmingham']",99500/99500,100.0%,1.000000,"['Queens', 'South Ozone Park', 'Howard Beach']",0.981416,1.000000,3
4,country,380358/380358,100.0%,1.000000,"['US', 'US', 'US']",63469/63469,100.0%,1.000000,"['US', 'US', 'US']",99500/99500,100.0%,1.000000,"['US', 'US', 'US']",1.000000,1.000000,3
5,house_number,344258/380358,90.5%,0.905089,"['3143', '16', '68']",62572/63469,98.6%,0.985867,"['224', '1521', '541-B']",97151/99500,97.6%,0.976392,"['135-25', '135-58', '158-22A']",0.955783,0.985867,3
6,kaggle380k_id,380358/380358,100.0%,1.000000,"['kaggle380k-000000', 'kaggle380k-000001', 'ka...",0/0,0%,0.000000,N/A,0/0,0%,0.000000,N/A,0.333333,1.000000,1
7,latitude,380140/380358,99.9%,0.999427,"[32.9338695, 32.945406, 32.9446173]",63469/63469,100.0%,1.000000,"[33.5623653, 33.58364, 33.5098]",99498/99500,100.0%,0.999980,"[40.66735, 40.6689814908178, 40.6601543540561]",0.999802,1.000000,3
8,longitude,380140/380358,99.9%,0.999427,"[-85.9704191, -85.953806, -85.954932]",63469/63469,100.0%,1.000000,"[-86.8307025, -86.77333, -86.85464]",99498/99500,100.0%,0.999980,"[-73.79692, -73.8216334221041, -73.84040060400...",0.999802,1.000000,3
9,map_url,345209/380358,90.8%,0.907590,['https://www.google.com/maps/place/Dairy+Quee...,0/0,0%,0.000000,N/A,99500/99500,100.0%,1.000000,['https://www.yelp.com/biz/the-quimby-queens?a...,0.635863,1.000000,2



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['address_line1', 'address_line2', 'categories', 'city', 'country', 'house_number', 'latitude', 'longitude', 'name', 'name_norm', 'postal_code', 'rating', 'source', 'state', 'street']


## Part 2: Entity Matching

In [5]:
# set up logging first

# Let's setup logging first
import logging

import os
os.makedirs('../output/logs', exist_ok=True)

# choose either default logging or debug logging

# Configure logging for INFO level
logging.basicConfig(
    level=logging.WARN,
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('../output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

### Step 1: Blocking

In [6]:
# create id columns
kaggle380k['id'] = kaggle380k['kaggle380k_id']
uber_eats['id'] = uber_eats['uber_eats_id']
yelp['id'] = yelp['yelp_id']

### kaggle380k -> uber_eats

In [8]:
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker

# Standard Blocking - First 3 characters of title
# Add title_prefix directly to the original dataframes
# kaggle380k['name_prefix'] = kaggle380k['name_norm'].astype(str).apply(lambda x: ''.join([word[:2].upper() for word in x.split()[:3]]))
# uber_eats['name_prefix'] = uber_eats['name_norm'].astype(str).apply(lambda x: ''.join([word[:2].upper() for word in x.split()[:3]]))

standard_blocker_k2u = StandardBlocker(
    kaggle380k, uber_eats,
    on=['city'],
    batch_size=1000,
    output_dir="../output/blocking-evaluation",
    id_column='id'
)

#standard_candidates_k2u = standard_blocker_k2u.materialize()

D:\Studium\Master\Semester 5\Team Project\workspace\venv3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
sn_blocker_k2u = SortedNeighbourhoodBlocker(
    kaggle380k, uber_eats,
    key='city',  # Sort by title
    window=20,     # Compare with 20 neighbors
    batch_size=1000,
    output_dir="../output/blocking-evaluation",
    id_column='id'
)
sn_candidates_k2u = sn_blocker_k2u.materialize()

In [9]:
token_blocker_k2u = TokenBlocker(
    kaggle380k, uber_eats,
    column='city',      # Tokenize titles
    batch_size=1000,
    output_dir="../output/blocking-evaluation",
    id_column='id',
    ngram_size=4,
    ngram_type='character'
)
#token_candidates_k2u = token_blocker_k2u.materialize()

In [9]:
embedding_blocker_k2u = EmbeddingBlocker(
    kaggle380k, uber_eats,
    text_cols=['city','name','state'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,          # Top 20 most similar
    batch_size=500,
    output_dir="../output/blocking-evaluation",
    id_column='id'
)
embedding_candidates_k2u = embedding_blocker_k2u.materialize()

KeyboardInterrupt: 

In [11]:
from PyDI.entitymatching import EntityMatchingEvaluator
from PyDI.io import load_csv

evaluator = EntityMatchingEvaluator()

# Create dictionaries of candidates for both dataset combinations
k2u_blocking_candidates = {
    'StandardBlocking': [standard_candidates_k2u, standard_blocker_k2u],
    'SortedNeighbourhoodBlocker': [sn_candidates_k2u, sn_blocker_k2u],
#    'TokenBlocking': [token_candidates_k2u,token_blocker_k2u],
    'EmbeddingBlocking': [embedding_candidates_k2u, embedding_blocker_k2u]
}

k2u_correspondences = load_csv(
    "ground_truth_to_label_df1_df2_labled.csv",
    name="k2u_test", header=None, names=['id1', 'id2', 'label'], add_index=False
)

k2u_results = []
for method_name, candidates in k2u_blocking_candidates.items():
    result = evaluator.evaluate_blocking_batched(blocker=candidates[1], test_pairs=k2u_correspondences, out_dir="../output/blocking-evaluation")
    result['method'] = method_name
    result['dataset'] = 'k2u'
    k2u_results.append(result)

k2u_best = max(k2u_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))
print(f"Best blocking for a2g: {k2u_best['method']} (PC: {k2u_best['pair_completeness']:.3f}, RR: {k2u_best['reduction_ratio']:.3f})")

Best blocking for a2g: StandardBlocking (PC: 0.960, RR: 0.999)


### kaggle380k -> yelp

In [12]:
standard_blocker_k2y = StandardBlocker(
    kaggle380k, yelp,
    on=['city'],
    batch_size=1000,
    output_dir="../output/blocking-evaluation",
    id_column='id'
)

standard_candidates_k2y = standard_blocker_k2y.materialize()

In [13]:
sn_blocker_k2y = SortedNeighbourhoodBlocker(
    kaggle380k, yelp,
    key='city',  # Sort by title
    window=20,     # Compare with 20 neighbors
    batch_size=1000,
    output_dir="../output/blocking-evaluation",
    id_column='id'
)
sn_candidates_k2y = sn_blocker_k2y.materialize()

In [14]:
token_blocker_k2y = TokenBlocker(
    kaggle380k, yelp,
    column='city',      # Tokenize titles
    batch_size=1000,
    output_dir="../output/blocking-evaluation",
    id_column='id',
    ngram_size=4,
    ngram_type='character'
)
#token_candidates_k2y = token_blocker_k2y.materialize()

In [15]:
embedding_blocker_k2y = EmbeddingBlocker(
    kaggle380k, yelp,
    text_cols=['city'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,          # Top 20 most similar
    batch_size=500,
    output_dir="../output/blocking-evaluation",
    id_column='id'
)
embedding_candidates_k2y = embedding_blocker_k2y.materialize()

In [17]:
evaluator = EntityMatchingEvaluator()

# Create dictionaries of candidates for both dataset combinations
k2y_blocking_candidates = {
    'StandardBlocking': [standard_candidates_k2y, standard_blocker_k2y],
    'SortedNeighbourhoodBlocker': [sn_candidates_k2y, sn_blocker_k2y],
#    'TokenBlocking': [token_candidates_k2y,token_blocker_k2y],
    'EmbeddingBlocking': [embedding_candidates_k2y, embedding_blocker_k2y]
}

k2y_correspondences = load_csv(
    "ground_truth_to_label_df1_df3_labled.csv",
    name="k2y_test", header=None, names=['id1', 'id2', 'label'], add_index=False
)

k2y_results = []
for method_name, candidates in k2y_blocking_candidates.items():
    result = evaluator.evaluate_blocking(candidates[0], k2y_correspondences,candidates[1], out_dir="../output/blocking-evaluation")
    result['method'] = method_name
    result['dataset'] = 'k2y'
    k2y_results.append(result)

k2y_best = max(k2y_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))
print(f"Best blocking for a2g: {k2y_best['method']} (PC: {k2y_best['pair_completeness']:.3f}, RR: {k2y_best['reduction_ratio']:.3f})")

Best blocking for a2g: StandardBlocking (PC: 0.931, RR: 0.999)


In [18]:
k2y_results

[{'pair_completeness': 0.930835734870317,
  'pair_quality': 1.1300032731611838e-05,
  'reduction_ratio': 0.99924472139062,
  'total_candidates': 28583988,
  'total_possible_pairs': 37845621000,
  'true_positives_found': 323,
  'total_true_pairs': 347,
  'evaluation_timestamp': '2025-10-31T12:45:07.490804',
  'output_files': ['output/blocking-evaluation\\blocking_evaluation_summary.json',
   'output/blocking-evaluation\\blocking_detailed_results.csv'],
  'method': 'StandardBlocking',
  'dataset': 'k2y'},
 {'pair_completeness': 0.01729106628242075,
  'pair_quality': 1.3009737788734868e-05,
  'reduction_ratio': 0.9999878138345253,
  'total_candidates': 461193,
  'total_possible_pairs': 37845621000,
  'true_positives_found': 6,
  'total_true_pairs': 347,
  'evaluation_timestamp': '2025-10-31T12:57:54.389092',
  'output_files': ['output/blocking-evaluation\\blocking_evaluation_summary.json',
   'output/blocking-evaluation\\blocking_detailed_results.csv'],
  'method': 'SortedNeighbourhoodBlo

In [19]:
k2u_results

[{'pair_completeness': 0.9596273291925466,
  'pair_quality': 1.2586332464917126e-05,
  'reduction_ratio': 0.9989830371946686,
  'total_candidates': 24550440,
  'total_possible_pairs': 24140941902,
  'true_positives_found': 309,
  'total_true_pairs': 322,
  'batches_processed': 24551,
  'evaluation_timestamp': '2025-10-31T09:00:00.630109',
  'output_files': ['output/blocking-evaluation\\blocking_evaluation_summary.json',
   'output/blocking-evaluation\\blocking_detailed_results.csv'],
  'method': 'StandardBlocking',
  'dataset': 'k2u'},
 {'pair_completeness': 0.034161490683229816,
  'pair_quality': 3.356943838329585e-05,
  'reduction_ratio': 0.999986426420256,
  'total_candidates': 327679,
  'total_possible_pairs': 24140941902,
  'true_positives_found': 11,
  'total_true_pairs': 322,
  'batches_processed': 328,
  'evaluation_timestamp': '2025-10-31T09:10:26.615283',
  'output_files': ['output/blocking-evaluation\\blocking_evaluation_summary.json',
   'output/blocking-evaluation\\blockin

# ML Entity Matching

In [56]:
from PyDI.io import load_parquet
from PyDI.entitymatching import FeatureExtractor
from PyDI.entitymatching import StringComparator, NumericComparator

# --------------------------------
# Prepare Data
# --------------------------------

# Define dataset paths
DATA_DIR = "../../datasets/"

# Load Kaggle 380k dataset
kaggle380k = load_parquet(
    DATA_DIR + "kaggle380k.parquet",
    name="kaggle380k",
)

# Load Uber Eats dataset  
uber_eats = load_parquet(
    DATA_DIR + "uber_eats.parquet",
    name="uber_eats",
)

# Load Yelp dataset
yelp = load_parquet(
    DATA_DIR + "yelp.parquet",
    name="yelp",
)

# create id columns
kaggle380k['id'] = kaggle380k['kaggle380k_id']
uber_eats['id'] = uber_eats['uber_eats_id']
yelp['id'] = yelp['yelp_id']

# Load ground truth correspondences
k2y_train = load_csv(
    "ground_truth_df1_df3_train.csv",
    name="ground_truth_df1_df3_train",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

k2y_test = load_csv(
    "ground_truth_df1_df3_test.csv",
    name="ground_truth_df1_df3_test",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

k2u_train = load_csv(
    "ground_truth_df1_df2_train.csv",
    name="ground_truth_df1_df2_train",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

k2u_test = load_csv(
    "ground_truth_df1_df2_test.csv",
    name="ground_truth_df1_df2_test",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

similarity_comparators = [
    # Name similarity features
    StringComparator("name_norm", similarity_function="jaro_winkler", preprocess=str.lower),
    StringComparator("name_norm", similarity_function="levenshtein", preprocess=str.lower),
    StringComparator("name_norm", similarity_function="cosine", preprocess=str.lower),
    StringComparator("name_norm", similarity_function="jaccard", preprocess=str.lower),

    # street name similarity
    StringComparator(
        column='street',
        similarity_function='jaccard', 
        preprocess=str.lower,
    ),

    # house number similarity
    NumericComparator(
        column='house_number',
        max_difference=2,
    ),

    # category similarity - supporting evidence
    StringComparator(
        column='categories',
        similarity_function='jaccard',
        preprocess=str.lower,
        list_strategy='concatenate' # Handle list attribute by concatenation
    ),
    StringComparator(
        column='categories',
        similarity_function='jaccard',
        preprocess=str.lower,
        list_strategy='best_match' # Handle list attribute by concatenation
    )
]

feature_extractor = FeatureExtractor(similarity_comparators)

# Extract features using FeatureExtractor
k2u_train_features = feature_extractor.create_features(
    kaggle380k, uber_eats, k2u_train[['id1', 'id2']], labels=k2u_train['label'], id_column='id'
)

# Extract features using FeatureExtractor
k2y_train_features = feature_extractor.create_features(
    kaggle380k, yelp, k2y_train[['id1', 'id2']], labels=k2y_train['label'], id_column='id'
)

# Prepare data for ML training
k2u_feature_columns = [col for col in k2u_train_features.columns if col not in ['id1', 'id2', 'label']]

k2u_X_train = k2u_train_features[k2u_feature_columns]
k2u_y_train = k2u_train_features['label']

k2y_feature_columns = [col for col in k2y_train_features.columns if col not in ['id1', 'id2', 'label']]

k2y_X_train = k2y_train_features[k2y_feature_columns]
k2y_y_train = k2y_train_features['label']

In [62]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, f1_score

# Define models and parameter grids
param_grids = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'class_weight': ['balanced', None]
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'penalty': ['l2'],
            'class_weight': ['balanced', None]
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.1, 0.2],
            'max_depth': [3, 5],
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'C': [0.1, 1.0, 10.0],
            'kernel': ['rbf', 'linear'],
            'class_weight': ['balanced', None]
        }
    }
}

# Use F1 score as the scoring metric (good for imbalanced data)
scorer = make_scorer(f1_score)
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"GridSearch setup: {len(param_grids)} models, F1 scoring, 5-fold CV")

# Train models using GridSearchCV
print(f"\n🚀 Training Models with GridSearchCV...")

grid_search_results = {}
best_overall_score = -1
best_overall_model = None
best_model_name = None

for model_name, config in param_grids.items():
    print(f"\nTraining {model_name}...")
    

    # Create GridSearchCV
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        scoring=scorer,
        cv=cv_folds,
        n_jobs=-1,  # Use all available cores
        verbose=0
    )
    
    # Fit GridSearchCV
    grid_search.fit(k2u_X_train, k2u_y_train)
    
    # Store results
    grid_search_results[model_name] = {
        'grid_search': grid_search,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        'best_estimator': grid_search.best_estimator_
    }
    
    print(f"  ✅ {model_name}: Best CV F1 = {grid_search.best_score_:.4f}")
    print(f"     Best params: {grid_search.best_params_}")
    
    # Track overall best model
    if grid_search.best_score_ > best_overall_score:
        best_overall_score = grid_search.best_score_
        k2u_best_overall_model = grid_search.best_estimator_
        best_model_name = model_name
            
print(f"\n🏆 Best Overall Model: {best_model_name} (CV F1: {best_overall_score:.4f})")

GridSearch setup: 4 models, F1 scoring, 5-fold CV

🚀 Training Models with GridSearchCV...

Training RandomForest...
  ✅ RandomForest: Best CV F1 = 0.9609
     Best params: {'class_weight': None, 'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}

Training LogisticRegression...
  ✅ LogisticRegression: Best CV F1 = 0.9461
     Best params: {'C': 10.0, 'class_weight': None, 'penalty': 'l2'}

Training GradientBoosting...
  ✅ GradientBoosting: Best CV F1 = 0.9506
     Best params: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 50}

Training SVM...
  ✅ SVM: Best CV F1 = 0.9463
     Best params: {'C': 1.0, 'class_weight': None, 'kernel': 'linear'}

🏆 Best Overall Model: RandomForest (CV F1: 0.9609)


In [58]:
grid_search_results = {}
best_overall_score = -1
best_overall_model = None
best_model_name = None

for model_name, config in param_grids.items():
    print(f"\nTraining {model_name}...")
    

    # Create GridSearchCV
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        scoring=scorer,
        cv=cv_folds,
        n_jobs=-1,  # Use all available cores
        verbose=0
    )
    
    # Fit GridSearchCV
    grid_search.fit(k2y_X_train, k2y_y_train)
    
    # Store results
    grid_search_results[model_name] = {
        'grid_search': grid_search,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        'best_estimator': grid_search.best_estimator_
    }
    
    print(f"  ✅ {model_name}: Best CV F1 = {grid_search.best_score_:.4f}")
    print(f"     Best params: {grid_search.best_params_}")
    
    # Track overall best model
    if grid_search.best_score_ > best_overall_score:
        best_overall_score = grid_search.best_score_
        k2y_best_overall_model = grid_search.best_estimator_
        best_model_name = model_name
            
print(f"\n🏆 Best Overall Model: {best_model_name} (CV F1: {best_overall_score:.4f})")


Training RandomForest...
  ✅ RandomForest: Best CV F1 = 0.9296
     Best params: {'class_weight': None, 'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 50}

Training LogisticRegression...
  ✅ LogisticRegression: Best CV F1 = 0.9071
     Best params: {'C': 10.0, 'class_weight': None, 'penalty': 'l2'}

Training GradientBoosting...
  ✅ GradientBoosting: Best CV F1 = 0.9200
     Best params: {'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100}

Training SVM...
  ✅ SVM: Best CV F1 = 0.9178
     Best params: {'C': 10.0, 'class_weight': None, 'kernel': 'rbf'}

🏆 Best Overall Model: RandomForest (CV F1: 0.9296)
